# Rubin DP2 Cutout Mosaics: HST-unresolved / Rubin-resolved COSMOS Sources

Builds RGB cutout mosaics for a **sample CSV** produced by the `make_sample_*.py`
scripts in `src/` (which share `src/cutout_samples.py`). Point `SAMPLE_NAME` at
whichever sample you want - the notebook does not define the selection itself, so
a new sample never requires editing this notebook.

Objects are HST-unresolved (COSMOS2020 FARMER `ACS_MU_CLASS = 2`) but
Rubin-resolved (`refExtendedness = 1`).

| sample | selection |
|---|---|
| `paper` | `dm > 0.05`, `20 < r < 22`, sorted by dm desc |
| `full`  | `r < 23`, no dm cut, sorted by dm desc |

`dm` = `r_psfMag - r_cModelMag`. Each sample CSV carries `obj_num` (1..N in
sorted order), which is printed on every stamp and resolves to RA/Dec through
the matching `sample_<name>.txt`.

**Run on the RSP.** `deep_coadd` pixel data lives only in the Rubin Butler repo.

### DP2 image API notes

DP2 (Science Pipelines v30) changed the image stack relative to DP1/DP0:

| | DP1 / DP0 (v29-) | **DP2 (v30)** |
|---|---|---|
| dataset type | `deepCoadd` | `deep_coadd` |
| storage class | `ExposureF` | `CellCoadd` |
| geometry | `lsst.geom` `Box2I`/`ExtentI` | `lsst.images` `Box`/`Interval` |
| sky->pixel | `exposure.getWcs().skyToPixel()` | `sky_projection.sky_to_pixel()` |
| patch lookup | `skymap.findTract(...)` | `butler.query_datasets(..., OVERLAPS POINT)` |

`image.array` is a plain numpy array; `.to_legacy()` is only needed for
`afw_display`.

## 1. Setup

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
from astropy.visualization import make_lupton_rgb

from lsst.daf.butler import Butler
from lsst.images import Box

## 2. Configuration

Set `SAMPLE_NAME` to the sample you want to render. `FILTER_SETS` controls which
colour composites are produced; each gets its own output folder, and because the
object order comes from the CSV, stamp *n* is the same object in every set.

In [ ]:
# --- Which sample to render --------------------------------------------------
SAMPLE_NAME = "paper"                # "paper" | "full" | any sample_<name>.csv

# Where sample_<name>.csv lives. These are tried in order, so the same notebook
# works on the RSP (CSV uploaded next to the notebook) and in the repo checkout.
SAMPLE_DIRS = [".", "outputs/cutout_samples"]

# --- Butler ------------------------------------------------------------------
BUTLER_REPO = "dp2"
COLLECTION = "dp2"
COADD_DATASET_TYPE = "deep_coadd"    # DP2/v30 name (DP1 and earlier: "deepCoadd")

# --- Colour composites: name -> (R, G, B) bands ------------------------------
# Rendered in this order; each produces its own folder of mosaic pages.
FILTER_SETS = {
    "gri": ("i", "r", "g"),
    "ugr": ("r", "g", "u"),
    "izy": ("y", "z", "i"),
}

# --- Output ------------------------------------------------------------------
OUTPUT_ROOT = "outputs/cutout_mosaics"

# --- Cutout geometry ---------------------------------------------------------
STAMP_ARCSEC = 10.0
PIXEL_SCALE = 0.2                    # arcsec/pixel, LSSTCam
HALF_PIX = int(round(STAMP_ARCSEC / PIXEL_SCALE)) // 2

# --- Mosaic layout -----------------------------------------------------------
GRID = 5                             # GRID x GRID stamps per page
N_OBJECTS = None                     # None = all; e.g. 25 for a quick look
LABEL_FONTSIZE = 15                  # stamp label size (bold)
TITLE_FONTSIZE = 20

# --- RGB rendering -----------------------------------------------------------
LUPTON_STRETCH = 5.0                 # tune with the test cell in section 6
LUPTON_Q = 8.0

print(f"Sample: {SAMPLE_NAME}")
print(f"Stamp:  {2 * HALF_PIX + 1} pix = {(2 * HALF_PIX + 1) * PIXEL_SCALE:.1f} arcsec")
print(f"Sets:   {', '.join(FILTER_SETS)}")

## 3. Connect to the Butler

In [ ]:
butler = Butler(BUTLER_REPO, collections=COLLECTION)
print(f"Connected to {BUTLER_REPO!r}, collection {COLLECTION!r}")

# Fail loudly here rather than partway through a mosaic run.
available = {dt.name for dt in butler.registry.queryDatasetTypes("*oadd*")}
if COADD_DATASET_TYPE in available:
    print(f"  dataset type {COADD_DATASET_TYPE!r}: OK")
else:
    raise RuntimeError(
        f"{COADD_DATASET_TYPE!r} not found in this collection. "
        f"Coadd-like types present: {sorted(available)}"
    )

## 4. Cutout helpers

The DP2 pattern, per tutorial 104_5:

1. `butler.query_datasets("deep_coadd", where="... patch.region OVERLAPS POINT(:ra, :dec)")`
   finds the overlapping patch - no manual skymap tract/patch arithmetic.
2. `butler.get("deep_coadd.sky_projection", ...)` fetches **only** the WCS
   component, so pixels are not read just to locate the object.
3. `Box.factory[y:y+1, x:x+1].padded(HALF_PIX)` builds the cutout box.
4. `butler.get(ref, parameters={"bbox": box})` reads just that sub-region.

In [ ]:
PATCH_QUERY = "band.name=:band AND patch.region OVERLAPS POINT(:ra, :dec)"


def find_coadd_ref(ra, dec, band):
    """Return the deep_coadd DatasetRef covering (ra, dec) in this band."""
    refs = butler.query_datasets(
        COADD_DATASET_TYPE,
        where=PATCH_QUERY,
        bind={"band": band, "ra": float(ra), "dec": float(dec)},
        with_dimension_records=True,
        order_by=["patch.tract"],
    )
    if not refs:
        raise LookupError(f"no {COADD_DATASET_TYPE} in band {band} at ({ra:.6f}, {dec:.6f})")
    return refs[0]


def cutout_box(ref, ra, dec, half_pix=HALF_PIX):
    """Build the cutout Box, reading only the sky_projection component."""
    sky_projection = butler.get(f"{COADD_DATASET_TYPE}.sky_projection", dataId=ref.dataId)
    xy = sky_projection.sky_to_pixel(SkyCoord(ra, dec, unit="deg"))
    return Box.factory[round(xy.y):round(xy.y) + 1, round(xy.x):round(xy.x) + 1].padded(half_pix)


def get_cutout(ra, dec, band, half_pix=HALF_PIX):
    """Fetch a single-band cutout centred on (ra, dec)."""
    ref = find_coadd_ref(ra, dec, band)
    return butler.get(ref, parameters={"bbox": cutout_box(ref, ra, dec, half_pix)})


def get_rgb_cutout(ra, dec, bands, half_pix=HALF_PIX,
                   stretch=LUPTON_STRETCH, Q=LUPTON_Q):
    """Lupton RGB composite for one position from a (R, G, B) band triple."""
    arrays = []
    for band in bands:
        ref = find_coadd_ref(ra, dec, band)
        cutout = butler.get(ref, parameters={"bbox": cutout_box(ref, ra, dec, half_pix)})
        # .array is already a plain ndarray; copy so the NaN fill stays local.
        arr = np.array(cutout.image.array, dtype=float)
        arr[~np.isfinite(arr)] = 0.0
        arrays.append(arr)

    # Bands can differ by a pixel near a patch edge; trim to the common shape.
    h = min(a.shape[0] for a in arrays)
    w = min(a.shape[1] for a in arrays)
    arrays = [a[:h, :w] for a in arrays]

    return make_lupton_rgb(*arrays, stretch=stretch, Q=Q)

## 5. Load the sample

In [ ]:
candidates = [os.path.join(d, f"sample_{SAMPLE_NAME}.csv") for d in SAMPLE_DIRS]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        f"No sample_{SAMPLE_NAME}.csv found. Looked in: {candidates}. "
        f"Generate it with src/make_sample_{SAMPLE_NAME}.py, or upload it next to "
        f"this notebook."
    )

df = pd.read_csv(csv_path)
if N_OBJECTS is not None:
    df = df.head(N_OBJECTS)

# obj_num is assigned by the sample script in sorted order; keep it as the label.
assert "obj_num" in df.columns, f"{csv_path} has no obj_num column - regenerate it"

print(f"Loaded {len(df):,} objects from {csv_path}")
print(f"  obj_num:     {df['obj_num'].min()} .. {df['obj_num'].max()}")
print(f"  r_cModelMag: {df['r_cModelMag'].min():.2f} .. {df['r_cModelMag'].max():.2f}")
print(f"  dm:          {df['r_psf_minus_cModel'].min():.3f} .. {df['r_psf_minus_cModel'].max():.3f}")
print(f"  fromBlend:   {df['detect_fromBlend'].mean():.1%}  isolated: {df['detect_isIsolated'].mean():.1%}")
print(f"  pages at {GRID}x{GRID}: {math.ceil(len(df) / GRID**2)} per filter set")
df.head()

## 6. Single-object test

**Run this before a full render.** Cheapest way to catch a bad collection,
dataset type, or box - and to tune `LUPTON_STRETCH` / `LUPTON_Q`, which depend on
the nJy flux scale. Check the object is centred.

In [ ]:
row = df.iloc[0]
print(f"#{row['obj_num']}  objectId {row['objectId']}  "
      f"ra={row['RA']:.6f} dec={row['DEC']:.6f}  r={row['r_cModelMag']:.2f}  "
      f"dm={row['r_psf_minus_cModel']:.3f}")

ref = find_coadd_ref(row["RA"], row["DEC"], "i")
print(f"dataId: {dict(ref.dataId.required)}")

arr = get_cutout(row["RA"], row["DEC"], "i").image.array
print(f"i-band stamp shape: {arr.shape}")
print(f"  flux percentiles (nJy): {np.nanpercentile(arr, [1, 50, 99]).round(2)}")

fig, axes = plt.subplots(1, 1 + len(FILTER_SETS), figsize=(5 * (1 + len(FILTER_SETS)), 5))
axes[0].imshow(arr, origin="lower", cmap="gray",
               vmin=np.nanpercentile(arr, 1), vmax=np.nanpercentile(arr, 99))
axes[0].set_title("i-band", fontsize=13)
for ax, (set_name, bands) in zip(axes[1:], FILTER_SETS.items()):
    ax.imshow(get_rgb_cutout(row["RA"], row["DEC"], bands), origin="lower")
    ax.set_title(f"{set_name}  ({'/'.join(bands)} -> RGB)", fontsize=13)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Generate the mosaics

One folder per filter set, same object order and `obj_num` throughout, so page
*p* slot *s* is the same source in `gri`, `ugr` and `izy`.

Stamps whose cutout fails are **skipped** - the grid closes up rather than
leaving a gap, so a page always holds `GRID**2` real stamps. Failures are
collected and reported at the end, and written to a text file for follow-up.

Labels carry `#obj_num`, r magnitude, `dm`, and a blend marker
(**B** = `detect_fromBlend`, **I** = `detect_isIsolated`).

In [ ]:
def stamp_label(row):
    flags = []
    if row["detect_fromBlend"]:
        flags.append("B")
    if row["detect_isIsolated"]:
        flags.append("I")
    flag_str = "".join(flags) if flags else "-"
    return (f"#{int(row['obj_num'])}  r={row['r_cModelMag']:.2f}  {flag_str}\n"
            f"dm={row['r_psf_minus_cModel']:.3f}")


def render_mosaics(sample, bands, set_name, output_dir):
    """Fetch every cutout, drop failures, then lay the survivors out on pages."""
    os.makedirs(output_dir, exist_ok=True)

    # Fetch first so failures can be skipped rather than left as blank cells.
    stamps, failures = [], []
    for _, row in sample.iterrows():
        try:
            rgb = get_rgb_cutout(row["RA"], row["DEC"], bands)
            stamps.append((row, rgb))
        except Exception as exc:
            failures.append({
                "obj_num": int(row["obj_num"]),
                "objectId": int(row["objectId"]),
                "RA": float(row["RA"]),
                "DEC": float(row["DEC"]),
                "filter_set": set_name,
                "error": str(exc),
            })

    n_pages = math.ceil(len(stamps) / GRID**2) if stamps else 0
    for page in range(n_pages):
        fig, axes = plt.subplots(GRID, GRID, figsize=(15, 15.5))
        axes = np.atleast_1d(axes).flatten()
        for ax in axes:
            ax.axis("off")

        for slot in range(GRID**2):
            idx = page * GRID**2 + slot
            if idx >= len(stamps):
                break
            row, rgb = stamps[idx]
            ax = axes[slot]
            ax.imshow(rgb, origin="lower")
            ax.set_title(stamp_label(row), fontsize=LABEL_FONTSIZE, fontweight="bold")

        fig.suptitle(
            f"HST-unresolved / Rubin-resolved COSMOS - {SAMPLE_NAME} - "
            f"{set_name} ({'/'.join(bands)}) - page {page + 1} of {n_pages}",
            fontsize=TITLE_FONTSIZE, fontweight="bold",
        )
        fig.tight_layout(rect=[0, 0, 1, 0.97])

        out_path = os.path.join(output_dir, f"mosaic_{set_name}_{page + 1:03d}.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"  saved {out_path}")
        plt.show()
        plt.close(fig)

    return stamps, failures

In [ ]:
all_failures = []

for set_name, bands in FILTER_SETS.items():
    print(f"\n=== {set_name}: {'/'.join(bands)} -> RGB ===")
    output_dir = os.path.join(OUTPUT_ROOT, SAMPLE_NAME, set_name)
    stamps, failures = render_mosaics(df, bands, set_name, output_dir)
    print(f"  {len(stamps)} stamps rendered, {len(failures)} skipped")
    all_failures.extend(failures)

## 8. Missing-cutout report

Every object has catalog data, so a failure here is worth understanding. This
writes the list to a text file alongside the mosaics.

In [ ]:
n_expected = len(df) * len(FILTER_SETS)
print(f"Attempted {n_expected} cutouts ({len(df)} objects x {len(FILTER_SETS)} filter sets)")
print(f"Failed:    {len(all_failures)}  ({len(all_failures) / n_expected:.2%})")

if all_failures:
    fail_df = pd.DataFrame(all_failures).sort_values(["obj_num", "filter_set"])

    # Objects that failed in every set are likely a coverage problem, not a
    # per-band one - worth separating when following these up.
    per_object = fail_df.groupby("obj_num")["filter_set"].nunique()
    all_sets = sorted(per_object[per_object == len(FILTER_SETS)].index)
    some_sets = sorted(per_object[per_object < len(FILTER_SETS)].index)
    print(f"  failed in ALL filter sets:  {len(all_sets)} objects {all_sets}")
    print(f"  failed in SOME filter sets: {len(some_sets)} objects {some_sets}")

    out_dir = os.path.join(OUTPUT_ROOT, SAMPLE_NAME)
    os.makedirs(out_dir, exist_ok=True)
    fail_path = os.path.join(out_dir, f"missing_cutouts_{SAMPLE_NAME}.txt")
    with open(fail_path, "w") as fh:
        fh.write(f"# Missing cutouts for sample '{SAMPLE_NAME}'\n")
        fh.write(f"# {len(all_failures)} of {n_expected} attempted "
                 f"({len(df)} objects x {len(FILTER_SETS)} filter sets)\n#\n")
        fh.write(fail_df.to_string(index=False))
        fh.write("\n")
    print(f"  wrote {fail_path}")

    display(fail_df)
else:
    print("  no missing cutouts")